In [1]:
import sqlite3

In [2]:
from datetime import datetime

In [ ]:
# Initialization
def get_connection():
    return sqlite3.connect("telehealth.db")

def init_db():
    conn = get_connection()
    conn.execute("PRAGMA foreign_keys = ON") 
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS patients (
                    patient_id INTEGER PRIMARY KEY,
                    first_name TEXT NOT NULL,
                    last_name TEXT NOT NULL,
                    age INTEGER,
                    phone TEXT NOT NULL,
                    email TEXT UNIQUE)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS appointments (
                    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    patient_id INTEGER,
                    date_time TEXT,
                    status TEXT,
                    FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
                    ON DELETE RESTRICT
                    ON UPDATE CASCADE )''') 
    conn.commit()
    conn.close()

init_db()


In [11]:
class Patient:
    def __init__(self, patient_id, first_name, last_name, age, phone, email):
        self.patient_id = patient_id
        self.first_name = first_name
        self.last_name = last_name
        self.age = age
        self.phone = phone
        self.email = email

    def save_to_db(self):
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO patients 
        (patient_id, first_name, last_name, age, phone, email)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            self.patient_id,
            self.first_name,
            self.last_name,
            self.age,
            self.phone,
            self.email
        ))

        conn.commit()
        conn.close()

#get a list of all patients    
    @staticmethod
    def list_all():
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM patients")
        rows = cursor.fetchall()
        conn.close()
        return rows
    
#get a specific patient 
    @staticmethod
    def get_by_id(patient_id):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM patients WHERE patient_id = ?", (patient_id,))
        row = cursor.fetchone()
        conn.close()
        return row


In [5]:
class Appointment:
    def __init__(self, appointment_id, patient_id, date_time, status="Scheduled"):
        self.appointment_id = appointment_id
        self.patient_id = patient_id
        self.date_time = date_time
        self.status = status

    def save_to_db(self):
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO appointments 
        (appointment_id, patient_id, date_time, status)
        VALUES (?, ?, ?, ?)
        """, (
            self.appointment_id,
            self.patient_id,
            self.date_time,
            self.status
        ))

        conn.commit()
        conn.close()
        
#get an appointement for a specific patient
    @staticmethod
    def for_patient(patient_id):
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM appointments WHERE patient_id = ?", (patient_id,))
        rows = cursor.fetchall()
        conn.close()
        return rows

In [ ]:
class Triage:
    SYMPTOM_RULES = {
        "fever": 2,
        "cough": 1,
        "chest pain": 3,
        "shortness of breath": 3,
        "headache": 1
    }

    @staticmethod
    def assess_symptoms(symptoms):
        severity = sum(Triage.SYMPTOM_RULES.get(s.lower(), 0) for s in symptoms)
        if severity >= 3:
            return "High"
        elif severity == 2:
            return "Moderate"
        return "Low"


class NotificationService:
    @staticmethod
    def send_reminder(patient, appointment):
        print(f"Reminder: {patient.first_name} {patient.last_name}, "
              f"appointment at {appointment.date_time}")

In [6]:
import csv

# --- Load Patients from CSV ---
with open("patients.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        p = Patient(
            int(row["patient_id"]),
            row["first_name"],
            row["last_name"],
            int(row["age"]),
            row["phone"],
            row["email"]
        )
        p.save_to_db()

# --- Load Appointments from CSV ---
with open("appointments.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        a = Appointment(
            int(row["appointment_id"]),
            int(row["patient_id"]),
            row["date_time"],
            row["status"]
        )
        a.save_to_db()

print("CSV data loaded into the database successfully!")


CSV data loaded into the database successfully!
